`The following topics covered`
1. PydanticAI Agent basics 
2. System prompts vs user prompts 
3. Structured (typed) outputs with `result_type`
4. Synchronous & async agent runs 
5. Mini-project: AI Resume Analyzer returning structured JSON 

# 1 ____ Install & Setup

In [1]:
%pip install pydantic-ai python-dotenv groq rich --quiet

Note: you may need to restart the kernel to use updated packages.


d:\AgenticAI\PydanticAI_Groq_AI_Agents\.venv\Scripts\python.exe: No module named pip


In [1]:
import os 
from dotenv import load_dotenv 
load_dotenv() 

# Make sure your Groq API key is set 
assert os.getenv("GROQ_API_KEY"),  "Set GROQ_API_KEYin .env!"
print("Environment loaded sucessfully!")

Environment loaded sucessfully!


# 2______ Your first PydanticAI Agent

In [4]:
## 2 - Your first PydanticAI agent 
from pydantic_ai import Agent 
agent = Agent(
    model="groq:llama-3.1-8b-instant", 
    system_prompt="""You are a helpful assistant. keep answers under 3 sentences."""
)

result = await agent.run("What is temperature in Lahore?")
print(result.output)

print(f"\n usage: {result.usage()}")


I don't have the current temperature data, but I can suggest checking a reliable weather source such as AccuWeather or the Pakistan Meteorological Department for the latest temperature information in Lahore. You can also search for "Lahore weather" or "Lahore temperature" online for updates.

 usage: RunUsage(input_tokens=54, output_tokens=60, requests=1)


# 3._____ Typed output: the killer feature

In [5]:
from pydantic import BaseModel, Field 
from typing import List 
from pydantic_ai import Agent 

class CountryFacts(BaseModel):
    country: str 
    capital: str 
    population_millions: float 
    language: List[str]
    fun_fact: str 

facts_agent = Agent(
    model = "groq:llama-3.1-8b-instant",
    output_type = CountryFacts, 
    system_prompt="Return ONLY valid JSON with country facts."
)

result = await facts_agent.run("Tell me about japan")

facts = result.output


print("Country", facts.country)
print("Capital", facts.capital)
print("Population", facts.population_millions, "M")
print("Language", ",".join(facts.language))
print("Fun fact", facts.fun_fact)

Country Japan
Capital Tokyo
Population 128.4 M
Language Japanese
Fun fact  Japan is home to the highest population density in the world and is known for its vibrant cities, rich culture, and cutting-edge technology.


# 4. --- Dynamic system prompt

In [6]:
from pydantic_ai import Agent, RunContext 
from dataclasses import dataclass 

@dataclass 
class UserContext:
    username: str 
    expertise_level: str

agent = Agent(
    model= "groq:llama-3.1-8b-instant",
    deps_type = UserContext
)

@agent.system_prompt
def prompt(ctx: RunContext[UserContext]) -> str:
    lvl = ctx.deps.expertise_level 
    # ctx = current run data 
    # deps = user data we passed 
    # extracting users level 
    return f"Tutor for {ctx.deps.username}. {'Simple explanation.' if lvl== 'beginer' else 'Technical explanaiton'}"
    # if user is beginner


#run 
for level in ["beginer", "expert"]:
    res = await agent.run(
        "What is list comprehension?",
        deps = UserContext("ikram", level)
    )
    print(f"\n=== {level} ===")
    print(res.output)



=== beginer ===
**What is List Comprehension?**

List comprehension is a powerful feature in Python (and some other programming languages) that allows you to create new lists from existing lists or other iterables using a concise syntax.

**Basic Syntax:**

```python
new_list = [expression for element in iterable]
```

Here's a breakdown of the components:

* `new_list`: This is the new list that will be created.
* `expression`: This is the operation you want to perform on each element in the iterable.
* `element`: This is the variable that will represent each element in the iterable.
* `iterable`: This is the list or other iterable that you want to process.

**Example:**

Suppose you have a list of numbers and you want to create a new list with the squares of each number:

```python
numbers = [1, 2, 3, 4, 5]
squares = [x**2 for x in numbers]
print(squares)  # Output: [1, 4, 9, 16, 25]
```

**Filtering:**

You can also use list comprehension to filter out elements based on a condition

# 5 ----- Multi-turn conversation (message history) 

In [9]:
from pydantic_ai import Agent 

chat_agent = Agent(
    model = "groq:llama-3.1-8b-instant", 
    system_prompt="You are a friendly Python tutor. Remember previous messages."
)

# Turn 1 
r1 = await chat_agent.run("My name is ikram and I am learning python.")
print("Turn 1:", r1.output)

# Turn 2 
r2 = await chat_agent.run(
    "What is my name and who am I?",
    message_history=r1.new_messages(),
)
print("Turn 2:", r2.output)

Turn 1: Hello Ikram. It's great to meet you! I'm glad to hear that you're learning Python. If you have any questions or need help with a specific topic, feel free to ask. 

What have you learned so far in Python? Do you have a specific topic or project that you're working on?
Turn 2: Your name is Ikram, and you are a Python learner. I'm your friendly Python tutor.


# 6 --- Async agents (for production apps)

In [10]:
from pydantic_ai import Agent 
from pydantic import BaseModel 
from typing import List  

class KeyPoints(BaseModel):
    topic: str
    points : List[str]


agent = Agent(
    model="groq:llama-3.1-8b-instant",
    output_type=KeyPoints,
    # This is the correct retry control (not model_settings)
    retries=3,
    system_prompt=(
        "You are a STRICT JSON generator. \n",
        "DO NOT use tools of search. \n",
        "DO NOT use function calls. \n",
        "Return ONLY valid JSON matching schema"
    ),
)


try:
    res = await agent.run("FastApi for building REST API's")

    kb = res.output

    print("Topic:", kb.topic)

    for i, p in enumerate(kb.points, 1):
        print(i, p)
except Exception as e:
    print("Failed safely:", e)

Topic: FastAPI for building REST API's
1 FastAPI is a modern, fast (high-performance), web framework for building APIs with Python 3.7+ based on standard Python type hints.
2 FastAPI uses the standard Python type hints for the API endpoint routes, request bodies, and response models.
3 It automatically generates API documentation for your API with support for interactive API interfaces.
4 It includes built-in support for WebSockets.
5 You can authenticate your API with JWT, OAuth2, and others using FastAPI.
6 It automatically handles CORS for you.
7 FastAPI is fully compatible with WSGI and ASGI servers.
8 It's perfect for building web APIs with Python 3.7+.
9 It supports GraphQL and OpenAPI v2.0, v3.0. It has async/await support.
10 It includes built-in support for automatic validation, automatic database schema creation, and more. It's easy to get started.


# Mini-Project: AI Resume Analyzer